# hERG blockade prediction — reproducible research notebook

**Scientific objective:** Compare molecular representations and classical ML classifiers without conflating feature choice with arbitrary hyperparameters.

**Workflow:** Data audit → scaffold-aware train/development/test → chemistry exploration → joint representation/model *coarse* tuning → focused tuning → threshold selection on development → one final test → export plan.

**Important:** This is a research notebook, **not** the assignment's `Part1_TEMPLATE.py`. Use the template's exact `load_split`, `featurize`, `build_model`, and ONNX contract for final submission. Our local scaffold split is a separate robustness experiment until reconciled with `load_split()`.

Each section contains its **question, hypothesis, method, and interpretation**. Do not tune based on the final test results. Out-of-fold tuning scores are selection scores, not unbiased final performance estimates.

## 0 · Settings & reproducibility

**Question:** What can we execute on a laptop CPU? **Method:** Default to a bounded, scientifically interpretable search. Set `RUN_EXTENSIONS=True` for optional additional molecular representations; set `RUN_FOCUSED=True` only after reviewing the coarse ranking. Do not overwrite results with a different split.

In [ ]:
from pathlib import Path
import os, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, StratifiedKFold, ParameterGrid, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, balanced_accuracy_score, precision_score, recall_score, f1_score
import sklearn, rdkit
RDLogger.DisableLog('rdApp.warning')
SEED=42
N_JOBS=2
RUN_EXTENSIONS=False        # MACCS / AtomPair optional
RUN_FOCUSED=False           # more expensive tuning after initial results
DATA_PATH=Path('hw1data.tab')
OUTPUT=Path('research_outputs'); OUTPUT.mkdir(exist_ok=True)
print('sklearn', sklearn.__version__, '| RDKit', rdkit.__version__)
assert DATA_PATH.exists(), 'Put hw1data.tab next to this notebook (or update DATA_PATH).'

## 1 · Data audit

**Question:** Are labels and molecular strings fit for modeling? **Hypothesis:** Invalid, duplicate or conflicting records can distort scores. **Method:** Audit full raw dataset; no label-dependent feature filtering.

In [ ]:
df=pd.read_csv(DATA_PATH, sep='\t')
assert {'Drug','Y'}.issubset(df.columns), f'Expected Drug and Y; found {df.columns.tolist()}'
assert df['Y'].dropna().isin([0,1]).all(), 'Y must be 0/1'
print('shape:', df.shape, '| label counts:'); display(df['Y'].value_counts(dropna=False).to_frame('count'))
print('missing:'); display(df.isna().sum().to_frame('missing'))
print('SMILES lengths:'); display(df['Drug'].astype(str).str.len().describe())
if df[['Drug','Y']].isna().any().any(): raise ValueError('Missing Drug/Y: review dataset before training')
molecules=[Chem.MolFromSmiles(s) for s in df['Drug']]
invalid=[i for i,m in enumerate(molecules) if m is None]
print('invalid SMILES:', len(invalid))
if invalid: raise ValueError('Invalid SMILES at rows: '+str(invalid[:20]))
df['Canonical_SMILES']=[Chem.MolToSmiles(m,canonical=True) for m in molecules]
print('duplicate canonical SMILES:', df['Canonical_SMILES'].duplicated().sum())
print('conflicting labels on identical canonical SMILES:', (df.groupby('Canonical_SMILES')['Y'].nunique()>1).sum())

## 2 · Scaffold groups and locked research split

**Question:** Does the model generalize beyond shared molecular scaffolds? **Hypothesis:** Scaffold-separated evaluation is harder but informative. **Design:** Match the *original notebook's* grouped split (80% group holdout followed by 12.5% within remaining groups). **Important:** `GroupShuffleSplit(test_size=...)` targets fractions of groups, not exactly fractions of molecules. Final test indices stay locked until section 9. The coursework's `load_split()` may use a different split; this cannot be assumed equivalent.

In [ ]:
def murcko(mol):
    return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol), canonical=True)
df['Scaffold']=[murcko(m) for m in molecules]
print('Unique scaffolds:', df['Scaffold'].nunique())
idx=np.arange(len(df)); y=df['Y'].to_numpy(dtype=int); groups=df['Scaffold'].to_numpy()
trainval_idx,test_idx=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED).split(idx,y,groups))
rel_train,rel_val=next(GroupShuffleSplit(n_splits=1,test_size=.125,random_state=SEED).split(trainval_idx, y[trainval_idx],groups[trainval_idx]))
train_idx=trainval_idx[rel_train]; val_idx=trainval_idx[rel_val]
for label,ii in [('Train',train_idx),('Dev validation',val_idx),('LOCKED test',test_idx)]:
    print(f'{label}: n={len(ii)}, positive fraction={y[ii].mean():.3f}, scaffolds={len(set(groups[ii]))}')
assert set(groups[train_idx]).isdisjoint(groups[val_idx])
assert set(groups[train_idx]).isdisjoint(groups[test_idx])
assert set(groups[val_idx]).isdisjoint(groups[test_idx])
assert min(np.bincount(y[train_idx]))>0 and min(np.bincount(y[val_idx]))>0

## 3 · Chemistry-driven exploratory analysis (train only)

**Question:** Which chemical properties differ across labels? **Hypothesis:** Lipophilicity, polarity, aromaticity, and related properties may contain predictive signal. **Caveat:** Descriptive association is not proof of causality, and these plots do not select the winning classifier.

In [ ]:
DESCRIPTOR_NAMES=['MolWt','MolLogP','TPSA','HBD','HBA','RotatableBonds','AromaticRings','FractionCSP3','FormalCharge','NitrogenCount']
def descriptors10(mols):
    arr=[]
    for m in mols:
        arr.append([Descriptors.MolWt(m),Descriptors.MolLogP(m),rdMolDescriptors.CalcTPSA(m),
        rdMolDescriptors.CalcNumHBD(m),rdMolDescriptors.CalcNumHBA(m),rdMolDescriptors.CalcNumRotatableBonds(m),
        rdMolDescriptors.CalcNumAromaticRings(m),rdMolDescriptors.CalcFractionCSP3(m),
        sum(a.GetFormalCharge() for a in m.GetAtoms()),sum(a.GetAtomicNum()==7 for a in m.GetAtoms())])
    return np.asarray(arr,dtype=np.float32)
X_desc=descriptors10(molecules)
chem=pd.DataFrame(X_desc[train_idx],columns=DESCRIPTOR_NAMES); chem['Y']=y[train_idx]
display(chem.groupby('Y')[DESCRIPTOR_NAMES].mean().T.round(3))
fig,axes=plt.subplots(1,3,figsize=(12,3))
for ax,col in zip(axes,['MolLogP','TPSA','AromaticRings']):
    for label in (0,1): ax.hist(chem.loc[chem.Y==label,col],bins=25,alpha=.45,label=f'Y={label}')
    ax.set_title(col);ax.legend()
plt.tight_layout();plt.show()

## 4 · Molecular representations (fixed-length)

**H1:** Morgan carries local connectivity information. **H2:** Global descriptors add complementary information. **H3:** Their benefit depends on classifier family. Start with Morgan binary radius 2 / 2048 bits, 10 descriptors, and hybrid. Later extensions (optional) test alternative representations. All feature construction below is deterministic, using no labels.

In [ ]:
def morgan_matrix(mols,radius=2,bits=2048,feature=False,count=False):
    kwargs={'radius':radius,'fpSize':bits}
    if feature: kwargs['atomInvariantsGenerator']=rdFingerprintGenerator.GetMorganFeatureAtomInvGen()
    gen=rdFingerprintGenerator.GetMorganGenerator(**kwargs)
    X=np.zeros((len(mols),bits),dtype=np.float32)
    for i,m in enumerate(mols):
        fp=gen.GetCountFingerprint(m) if count else gen.GetFingerprint(m)
        DataStructs.ConvertToNumpyArray(fp,X[i])
    return X
X_morgan=morgan_matrix(molecules)
feature_sets={'Morgan':X_morgan,'Descriptors10':X_desc,'Hybrid':np.hstack([X_morgan,X_desc]).astype(np.float32)}
for name,X in feature_sets.items(): print(name,X.shape,'finite?',np.isfinite(X).all())

## 5 · Fair *coarse* screening: model × representation × modest hyperparameter search

**Question:** Is a feature representation best across classifiers, or is there an interaction? **Method:** Tune each of the 12 combinations with the **same grouped CV folds**, using small model-appropriate grids. Do not compare untuned SVM against tuned RF. Training data only is used in this phase; validation is reserved for later model-development checks.

**Important:** Identical numbers of candidate settings do not give perfectly equal search effort across algorithms. The screen finds *promising candidates*, not a universal winner. `StandardScaler` is fitted inside each CV fold for LR/SVM; fingerprints are not scaled but descriptors are. Tree classifiers use raw features.

In [ ]:
cv=GroupKFold(n_splits=3)
SCORING='roc_auc'
def build_estimator(model_name,rep_name,n_features):
    if model_name in ('Logistic Regression','SVM-RBF'):
        if rep_name=='Descriptors10': pre=StandardScaler()
        elif rep_name=='Hybrid':
            pre=ColumnTransformer([('desc',StandardScaler(),list(range(n_features-10,n_features)))],remainder='passthrough')
        else: pre='passthrough'
        clf=(LogisticRegression(max_iter=2000,random_state=SEED) if model_name=='Logistic Regression'
             else SVC(kernel='rbf',probability=True,random_state=SEED))
        return Pipeline([('pre',pre),('model',clf)])
    if model_name=='Random Forest': return RandomForestClassifier(random_state=SEED,n_jobs=N_JOBS)
    if model_name=='Extra Trees': return ExtraTreesClassifier(random_state=SEED,n_jobs=N_JOBS)
    raise ValueError(model_name)
GRIDS={
    'Logistic Regression':{'model__C':[.03,.3,3.0], 'model__penalty':['l2']},
    'SVM-RBF':{'model__C':[.1,1,10], 'model__gamma':['scale',.01]},
    'Random Forest':{'n_estimators':[160], 'min_samples_leaf':[1,3,5], 'max_features':['sqrt',.2]},
    'Extra Trees':{'n_estimators':[160], 'min_samples_leaf':[1,3,5], 'max_features':['sqrt',.2]},
}
MODEL_NAMES=list(GRIDS)
def coarse_search(reps,models=MODEL_NAMES):
    from sklearn.model_selection import GridSearchCV
    records=[]; searches={}
    for rep_name,X in reps.items():
        for name in models:
            tic=time.time(); est=build_estimator(name,rep_name,X.shape[1]); key=(rep_name,name)
            search=GridSearchCV(est,GRIDS[name],scoring=SCORING,cv=cv,n_jobs=1,refit=True,error_score='raise')
            search.fit(X[train_idx],y[train_idx],groups=groups[train_idx])
            searches[key]=search
            records.append({'Representation':rep_name,'Model':name,'Mean_CV_AUROC':search.best_score_,
                            'CV_SD':search.cv_results_['std_test_score'][search.best_index_],
                            'Settings':search.best_params_,'Seconds':round(time.time()-tic,1)})
            print(rep_name,name,'AUROC',round(search.best_score_,4),'seconds',records[-1]['Seconds'])
    return pd.DataFrame(records).sort_values('Mean_CV_AUROC',ascending=False).reset_index(drop=True),searches
coarse_table,coarse_searches=coarse_search(feature_sets)
display(coarse_table)
coarse_table.to_csv(OUTPUT/'coarse_cv.csv',index=False)

### Interpret the controlled comparison

For each model, compute Hybrid minus Morgan score: this is an *exploratory CV difference*, not a significance test. If rankings change across model families, that is direct evidence of **model–representation interaction in these experiments**. Differences smaller than CV variability should not be overinterpreted.

In [ ]:
pivot=coarse_table.pivot(index='Model',columns='Representation',values='Mean_CV_AUROC')
display(pivot.round(4))
if {'Hybrid','Morgan'}.issubset(pivot.columns):
    display((pivot['Hybrid']-pivot['Morgan']).rename('Hybrid_minus_Morgan_CV_AUROC').to_frame().round(4))

## 6 · Optional extension: alternative fingerprints and richer descriptors

**Question:** Did our original three representations miss complementary information? **Hypotheses:** (a) pharmacophore-feature Morgan Count captures different local patterns; (b) MACCS keys may offer a compact, fast baseline; (c) atom-pair features represent longer-range topology; (d) more descriptors may add signal but also noise. By default **disabled** to keep the CPU budget manageable. After enabling, these compete through the **same coarse tuner**; do not compare their untuned scores against tuned candidates.

For extended descriptors, nonfinite values are replaced using a train-fold-fitted median imputer; feature selection must likewise occur inside CV. The extension below uses MACCS + Feature Morgan Count, avoiding any data-dependent preprocessing beyond scaling as currently written.

In [ ]:
extension_results=None
if RUN_EXTENSIONS:
    X_maccs=np.zeros((len(molecules),167),dtype=np.float32)
    for i,m in enumerate(molecules): DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m),X_maccs[i])
    X_fc=morgan_matrix(molecules,radius=2,bits=2048,feature=True,count=True)
    # Build as named hybrid arrays so the last 10 columns always contain the descriptors.
    extra={'MACCS':X_maccs,'FeatureCount':X_fc,'FeatureCount_Hybrid':np.hstack([X_fc,X_desc]).astype(np.float32)}
    # build_estimator identifies only 'Descriptors10' and 'Hybrid' for scaling.
    # Rename FeatureCount_Hybrid to Hybrid temporarily via a generalized helper before tuning.
    # Trees are evaluated for all extras; LR/SVM can use MACCS and FeatureCount as fingerprint-only inputs.
    for rep in extra:
        if rep.endswith('_Hybrid'): continue
        sub, _=coarse_search({rep:extra[rep]})
        extension_results=sub if extension_results is None else pd.concat([extension_results,sub],ignore_index=True)
    if extension_results is not None:
        display(extension_results.sort_values('Mean_CV_AUROC',ascending=False))
        extension_results.to_csv(OUTPUT/'optional_extension_cv.csv',index=False)
else:
    print('Skipped extensions. Change RUN_EXTENSIONS=True in settings to run them.')

## 7 · Development validation: compare shortlisted configurations once

**Question:** Do top CV candidates hold up on scaffolds excluded from training CV? **Method:** Compare up to four distinct top-scoring model/representation pairs on the held-out **development validation** split. **Caution:** Once you inspect and use this validation to choose models, it is no longer an unbiased final evaluation set. Do not repeatedly change search spaces based on validation outcomes.

Use this as a limited confirmation/sanity check; the one final test remains untouched.

In [ ]:
def metric_row(y_true,p,threshold=.5):
    pred=(p>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return {'AUROC':roc_auc_score(y_true,p),'AUPRC':average_precision_score(y_true,p),
            'Sensitivity':tp/(tp+fn) if (tp+fn) else np.nan,
            'Specificity':tn/(tn+fp) if (tn+fp) else np.nan,
            'Precision':tp/(tp+fp) if (tp+fp) else np.nan,
            'BalancedAccuracy':balanced_accuracy_score(y_true,pred)}
selected=[]
for row in coarse_table.itertuples(index=False):
    key=(row.Representation,row.Model)
    if key not in selected: selected.append(key)
    if len(selected)==4:break
valid_records=[]
for rep,name in selected:
    X=feature_sets[rep]; fitted=coarse_searches[(rep,name)].best_estimator_
    p=fitted.predict_proba(X[val_idx])[:,1]
    valid_records.append({'Representation':rep,'Model':name,**metric_row(y[val_idx],p)})
validation_table=pd.DataFrame(valid_records).sort_values('AUROC',ascending=False)
display(validation_table.round(4));validation_table.to_csv(OUTPUT/'development_validation.csv',index=False)

## 8 · Optional focused tuning (TRAIN only)

**Question:** Can extra tuning meaningfully improve the leading candidates? **Method:** A moderate randomized search for the top two *coarse-CV* pairs (not validation-selected ones). This stage does not touch final test. We choose model candidates using train CV scores, not test data. Note: refined CV scores have selection bias, so do not report them as independent generalization estimates.

In [ ]:
focused_searches={}
if RUN_FOCUSED:
    focused_grids={
      'Logistic Regression':{'model__C':np.logspace(-3,2,12),'model__penalty':['l2']},
      'SVM-RBF':{'model__C':np.logspace(-2,2,8),'model__gamma':['scale',.0001,.001,.01,.1]},
      'Random Forest':{'n_estimators':[150,250,400],'min_samples_leaf':[1,2,4,7],'max_features':['sqrt',.1,.25],'max_depth':[None,16,32]},
      'Extra Trees':{'n_estimators':[150,250,400],'min_samples_leaf':[1,2,4,7],'max_features':['sqrt',.1,.25],'max_depth':[None,16,32]},
    }
    for rep,name in selected[:2]:
        X=feature_sets[rep]
        tuner=RandomizedSearchCV(build_estimator(name,rep,X.shape[1]),focused_grids[name],n_iter=12,
                                 scoring=SCORING,cv=cv,refit=True,random_state=SEED,n_jobs=1,error_score='raise')
        tuner.fit(X[train_idx],y[train_idx],groups=groups[train_idx]);focused_searches[(rep,name)]=tuner
        print(rep,name,'focused AUROC',round(tuner.best_score_,4),'params',tuner.best_params_)
else:print('Skipped focused tuning. Set RUN_FOCUSED=True for additional training-CV search.')

## 9 · Choose an operational threshold without touching the final test

**Question:** How can we prioritize blocker sensitivity while keeping specificity practical? **Hypothesis:** A threshold below 0.5 may improve recall, but this must be checked. **Method:** Choose the best *train-CV* candidate (or its focused version), get probabilities on the development validation set, then choose threshold maximizing sensitivity subject to specificity ≥ 0.73 (0.70 assignment constraint plus 0.03 buffer). This uses validation for threshold **selection**; its resulting metrics are descriptive/tuning metrics, **not** final performance. The untouched test is used only after threshold is frozen.

In [ ]:
winning_rep,winning_name=selected[0]
chosen_search=focused_searches.get((winning_rep,winning_name),coarse_searches[(winning_rep,winning_name)])
chosen_estimator=clone(chosen_search.best_estimator_)
Xwin=feature_sets[winning_rep]
chosen_estimator.fit(Xwin[train_idx],y[train_idx])
p_val=chosen_estimator.predict_proba(Xwin[val_idx])[:,1]
threshold_rows=[]
for t in np.linspace(.05,.95,181):
    threshold_rows.append({'Threshold':float(t),**metric_row(y[val_idx],p_val,t)})
threshold_table=pd.DataFrame(threshold_rows)
feasible=threshold_table[threshold_table['Specificity']>=.73]
if len(feasible):
    best_threshold=feasible.sort_values(['Sensitivity','Specificity'],ascending=False).iloc[0]['Threshold']
    print('Threshold chosen for high sensitivity with specificity>=0.73:',round(best_threshold,3))
else:
    best_threshold=.5;print('No feasible threshold; default to 0.5 and document constraint failure.')
print('Winning candidate:',winning_rep,winning_name)
display(threshold_table.loc[(threshold_table.Threshold-best_threshold).abs().nsmallest(1).index].round(4))
fig,ax=plt.subplots(figsize=(6,4));ax.plot(threshold_table.Threshold,threshold_table.Sensitivity,label='Sensitivity');ax.plot(threshold_table.Threshold,threshold_table.Specificity,label='Specificity');ax.axvline(best_threshold,ls='--',label='selected threshold');ax.legend();ax.set_xlabel('Decision threshold');plt.show()

## 10 · Single final evaluation on LOCKED research test

**Only execute after the pipeline and threshold are frozen.** Fit once on train + development validation, then assess once on previously unseen test scaffolds. Do not change the model after viewing final results. This test is the **research scaffold test**, not necessarily the official assignment split or hidden competition test.

`RUN_FINAL_TEST` defaults to `False` to avoid accidental early evaluation.

In [ ]:
RUN_FINAL_TEST=False    # switch True only when ready to lock final configuration
if RUN_FINAL_TEST:
    final_est=clone(chosen_search.best_estimator_)
    fit_idx=np.concatenate([train_idx,val_idx])
    final_est.fit(Xwin[fit_idx],y[fit_idx])
    test_p=final_est.predict_proba(Xwin[test_idx])[:,1]
    final_result=metric_row(y[test_idx],test_p,float(best_threshold))
    print('LOCKED RESEARCH TEST (do not tune after seeing):');display(pd.DataFrame([final_result]).round(4))
    (OUTPUT/'research_choice.json').write_text(json.dumps({'representation':winning_rep,'model':winning_name,
       'threshold':float(best_threshold),'best_params':chosen_search.best_params_},indent=2,default=str))
else:print('Final test is locked. Set RUN_FINAL_TEST=True only after freezing all decisions.')

## 11 · Submission integration checklist

This notebook does **not** directly generate a valid assignment ONNX because the provided template, `load_split()` implementation, `TestModel.py`, and pinned environment files were not supplied with the uploaded notebook. **Do not** assume a fitted sklearn pipeline can be exported safely without matching the template's inference contract.

1. Open `Part1_TEMPLATE.py` and inspect `load_split()` — run model selection with the **official train/validation splits** as required, separately from the scaffold sensitivity analysis above.
2. Implement one deterministic `featurize()` matching the winning feature order; it must accept valid SMILES of **up to 200 characters** (do not pad/truncate to string length).
3. Apply exactly the same feature construction at training and web-app inference; fit all learned transforms only on training data and preserve them in the inference/export pipeline.
4. `build_model()` must use fixed, chosen hyperparameters and class mapping (Y=1 means blocker); apply frozen threshold consistently at inference, or return probability as the template expects.
5. Export ONNX in the prescribed way; verify the exact output structure and probability mapping with `TestModel.py` and compare against web-app predictions.
6. Record uncertainty: external hidden competition performance cannot be inferred from the development scores. Training must finish in **<30 minutes on laptop CPU**, ONNX file **≤50 MB**, and no CUDA/MPS. Record use of an LLM in the README.

**Scientific note:** To estimate the entire *selection process* without optimism, use nested group CV (representation selection, model choice, hyperparameters, and threshold *inside* each outer fold); this notebook instead gives a practical resource-bounded development process with a single untouched final scaffold test.